# 02 — Simulator and Gymnasium environment

Loads **preprocessing artifacts from notebook 01** (does not refit them).

Builds the learner-response simulator, validates the environment, and writes **synthetic** trajectories with provenance.

**Expected outputs:**
- `synthetic/sim_train_trajectories.csv` (+ val/test)
- `synthetic/*_provenance.json`
- `synthetic/simulator_assumptions.json`
- `synthetic/action_catalogue.json`


In [ ]:
# === USER CONFIG ===
RUN_IN_COLAB = True
DRIVE_SUBDIR = "AdaptiveLearnSE/c4_runs"
C4_ROOT = None  # defaults to parent of experiments/


In [ ]:
from pathlib import Path
import sys
import subprocess

if C4_ROOT is None:
    C4_ROOT = Path("..").resolve()
sys.path.insert(0, str(C4_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(C4_ROOT / "requirements-training.txt")])

from training.paths import resolve_output_root, stage_dirs
from training.colab_bootstrap import mount_drive
from training.config import EnvConfig
from training.pipeline import run_simulator_stage, load_simulator_from_artifacts, require_previous_stage
from training.environment import make_env
from gymnasium.utils.env_checker import check_env

if RUN_IN_COLAB:
    try:
        mount_drive()
    except Exception as e:
        print(e)

OUT = resolve_output_root(DRIVE_SUBDIR if RUN_IN_COLAB else None, prefer_drive=RUN_IN_COLAB)
DIRS = stage_dirs(OUT)
ARTIFACTS = require_previous_stage(
    DIRS["artifacts"] / "response_model.joblib",
    "01_data_preparation",
    "Run notebook 01 first.",
).parent
print("Artifacts:", ARTIFACTS)


In [ ]:
env_cfg = EnvConfig(include_simulation_only_actions=False)
sim_info = run_simulator_stage(ARTIFACTS, output_dir=DIRS["synthetic"], env_config=env_cfg, seed=42)
print(sim_info)

sim = load_simulator_from_artifacts(ARTIFACTS, env_cfg)
env = make_env(sim, env_cfg)
check_env(env, skip_render_check=True)
obs, info = env.reset(seed=0)
print("obs shape", obs.shape, "n_actions", env.action_space.n)
print("action catalogue sample", env.action_catalogue()[:3])
print("Fitted vs assumed:", sim.assumptions()["fitted_from_data"], list(sim.assumptions()["assumed_not_fitted"]))


Synthetic files are **not** ASSISTments rows. Do not use them as evidence of SE learning gains.
